<a href="https://colab.research.google.com/github/AndrijaM06/car-price-prediction/blob/main/01_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EDA: Predikcija cene polovnih automobila

Istraživačka analiza skupa podataka `cars.csv` pre čišćenja i treniranja regresionog modela.

## Učitavanje podataka i prvi pregled

In [4]:
from pathlib import Path
import pandas as pd

url = "https://raw.githubusercontent.com/AndrijaM06/car-price-prediction/main/data/cars.csv"
df = pd.read_csv(url)
df.head()

,make,model,priceUSD,year,condition,mileage(kilometers),fuel_type,volume(cm3),color,transmission,drive_unit,segment
0,mazda,2,5500,2008,with mileage,162000.0,petrol,1500.0,burgundy,mechanics,front-wheel drive,B
1,mazda,2,5350,2009,with mileage,120000.0,petrol,1300.0,black,mechanics,front-wheel drive,B
2,mazda,2,7000,2009,with mileage,61000.0,petrol,1500.0,silver,auto,front-wheel drive,B
3,mazda,2,3300,2003,with mileage,265000.0,diesel,1400.0,white,mechanics,front-wheel drive,B
4,mazda,2,5200,2008,with mileage,97183.0,diesel,1400.0,gray,mechanics,front-wheel drive,B


In [5]:
df.sample(5, random_state=42)

,make,model,priceUSD,year,condition,mileage(kilometers),fuel_type,volume(cm3),color,transmission,drive_unit,segment
19270,mitsubishi,carisma,2050,1999,with mileage,250000.0,petrol,1800.0,blue,mechanics,front-wheel drive,M
25927,ford,fusion,4500,2006,with mileage,160000.0,petrol,1400.0,burgundy,mechanics,front-wheel drive,M
23388,mercedes-benz,e-klass,3500,1999,with mileage,485000.0,diesel,2900.0,blue,mechanics,rear drive,E
53189,bmw,x3,21000,2013,with mileage,159000.0,diesel,2000.0,black,auto,NaN,J
34058,renault,megane,3990,2002,with mileage,331700.0,diesel,1900.0,other,mechanics,front-wheel drive,C


## Osnovna struktura skupa podataka

In [6]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

Rows: 56244
Columns: 12


In [7]:
df.columns.tolist()

['make',
 'model',
 'priceUSD',
 'year',
 'condition',
 'mileage(kilometers)',
 'fuel_type',
 'volume(cm3)',
 'color',
 'transmission',
 'drive_unit',
 'segment']

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56244 entries, 0 to 56243
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   make                 56244 non-null  object 
 1   model                56244 non-null  object 
 2   priceUSD             56244 non-null  int64  
 3   year                 56244 non-null  int64  
 4   condition            56244 non-null  object 
 5   mileage(kilometers)  56244 non-null  float64
 6   fuel_type            56244 non-null  object 
 7   volume(cm3)          56197 non-null  float64
 8   color                56244 non-null  object 
 9   transmission         56244 non-null  object 
 10  drive_unit           54339 non-null  object 
 11  segment              50953 non-null  object 
dtypes: float64(2), int64(2), object(8)
memory usage: 5.1+ MB


## Razumevanje značenja kolona

| Kolona | Namena |
|---|---|
| make | marka automobila |
| model | model automobila |
| priceUSD | cena automobila u dolarima (ciljna promenljiva) |
| year | godina proizvodnje |
| condition | stanje automobila |
| mileage(kilometers) | kilometraža |
| fuel_type | tip goriva |
| volume(cm3) | zapremina motora |
| color | boja automobila |
| transmission | tip menjača |
| drive_unit | tip pogona |
| segment | segment automobila |


## Analiza ciljne promenljive (`priceUSD`)

In [9]:
df["priceUSD"].dtype

dtype('int64')

In [10]:
df["priceUSD"].describe()

,priceUSD
count,56244.000000
mean,7415.456440
std,8316.959261
min,48.000000
25%,2350.000000
50%,5350.000000
75%,9807.500000
max,235235.000000


In [11]:
(df["priceUSD"] <= 0).sum()

np.int64(0)

**Zaključak:** `priceUSD` je već numerička kolona (int64), nema nula ili negativnih vrednosti. Maksimalna vrednost deluje kao ekstremna, ali ne nužno pogrešna — ima nekoliko veoma skupih automobila. Ovo ćemo dodatno proveriti u fazi čišćenja.

## Provera nedostajućih vrednosti

In [12]:
missing_values = df.isna().sum()
missing_values[missing_values > 0]

,0
volume(cm3),47
drive_unit,1905
segment,5291


**Zaključak:** Nedostajuće vrednosti postoje u tri kolone:
- `volume(cm3)` — 47 nedostajućih vrednosti
- `drive_unit` — 1905 nedostajućih vrednosti
- `segment` — 5291 nedostajućih vrednosti (najveći broj)

Ciljna promenljiva `priceUSD` nema nedostajućih vrednosti, što je dobro.

## Provera duplikata

In [13]:
df.duplicated().sum()

np.int64(87)

**Zaključak:** Postoji 87 potpuno identičnih redova. Ovo ćemo ukloniti u fazi čišćenja.

## Analiza numeričkih kolona

In [14]:
numeric_columns = [
    "priceUSD",
    "year",
    "mileage(kilometers)",
    "volume(cm3)",
]

df[numeric_columns].describe()

,priceUSD,year,mileage(kilometers),volume(cm3)
count,56244.000000,56244.000000,5.624400e+04,56197.000000
mean,7415.456440,2003.454840,2.443956e+05,2104.860615
std,8316.959261,8.144247,3.210307e+05,959.201633
min,48.000000,1910.000000,0.000000e+00,500.000000
25%,2350.000000,1998.000000,1.370000e+05,1600.000000
50%,5350.000000,2004.000000,2.285000e+05,1996.000000
75%,9807.500000,2010.000000,3.100000e+05,2300.000000
max,235235.000000,2019.000000,9.999999e+06,20000.000000


In [15]:
df["year"].describe()

,year
count,56244.000000
mean,2003.454840
std,8.144247
min,1910.000000
25%,1998.000000
50%,2004.000000
75%,2010.000000
max,2019.000000


In [16]:
(df["year"] < 1970).sum()

np.int64(124)

**Zaključak (year):** Opseg godina je 1910–2019. Postoji 124 automobila starijih od 1970. godine — ovo su verovatno validni, ali retki, oldtimer automobili. Treba odlučiti da li ih zadržati ili tretirati kao outliere.

In [17]:
df["mileage(kilometers)"].describe()

,mileage(kilometers)
count,5.624400e+04
mean,2.443956e+05
std,3.210307e+05
min,0.000000e+00
25%,1.370000e+05
50%,2.285000e+05
75%,3.100000e+05
max,9.999999e+06


In [18]:
(df["mileage(kilometers)"] > 1_000_000).sum()

np.int64(364)

In [19]:
df[df["mileage(kilometers)"] > 1_000_000]["mileage(kilometers)"].describe()

,mileage(kilometers)
count,3.640000e+02
mean,3.134361e+06
std,2.250612e+06
min,1.100000e+06
25%,1.228642e+06
50%,2.800000e+06
75%,3.700000e+06
max,9.999999e+06


**Zaključak (mileage):** Maksimalna vrednost je skoro 10.000.000 km, što je fizički nemoguće za automobil. Postoji 364 reda sa kilometražom preko 1.000.000 km — ovo su skoro sigurno pogrešno unete vrednosti (npr. korisnik je greškom uneo dodatne nule). Ove redove ćemo morati da tretiramo kao nevalidne u fazi čišćenja.

In [20]:
df["volume(cm3)"].describe()

,volume(cm3)
count,56197.000000
mean,2104.860615
std,959.201633
min,500.000000
25%,1600.000000
50%,1996.000000
75%,2300.000000
max,20000.000000


**Zaključak (volume):** Opseg (500–20000 cm³) deluje realno za većinu, ali maksimalna vrednost od 20000 cm³ je neuobičajeno velika za putnički automobil — treba proveriti kao potencijalni outlier.

## Analiza kategorijskih kolona

In [21]:
categorical_columns = [
    "make",
    "model",
    "condition",
    "fuel_type",
    "color",
    "transmission",
    "drive_unit",
    "segment",
]

for column in categorical_columns:
    print(column)
    print(df[column].nunique())
    print("-" * 40)

make
96
----------------------------------------
model
1034
----------------------------------------
condition
3
----------------------------------------
fuel_type
3
----------------------------------------
color
13
----------------------------------------
transmission
2
----------------------------------------
drive_unit
4
----------------------------------------
segment
9
----------------------------------------


In [22]:
for column in ["condition", "fuel_type", "transmission", "drive_unit", "segment"]:
    print(column)
    print(df[column].value_counts(dropna=False))
    print("-" * 40)

condition
condition
with mileage    55278
with damage       512
for parts         454
Name: count, dtype: int64
----------------------------------------
fuel_type
fuel_type
petrol        36405
diesel        19792
electrocar       47
Name: count, dtype: int64
----------------------------------------
transmission
transmission
mechanics    36056
auto         20188
Name: count, dtype: int64
----------------------------------------
drive_unit
drive_unit
front-wheel drive             38016
rear drive                     6836
all-wheel drive                5890
part-time four-wheel drive     3597
NaN                            1905
Name: count, dtype: int64
----------------------------------------
segment
segment
D      12605
C      10617
J       8629
M       6313
E       6274
NaN     5291
B       4393
F        896
S        765
A        461
Name: count, dtype: int64
----------------------------------------


**Zaključak (kategorijske kolone):**
- `condition` ima 3 vrednosti: `with mileage` (dominantna, 55278), `with damage`, `for parts`. Vrednosti `with damage` i `for parts` opisuju automobile koji nisu u standardnom stanju — ovo može biti korisna karakteristika ili razlog za izdvajanje tih redova.
- `fuel_type` ima 3 vrednosti, `electrocar` je vrlo redak (47 redova).
- `transmission` ima samo 2 vrednosti — čist, binaran atribut.
- `drive_unit` ima 4 vrednosti + nedostajuće vrednosti.
- `segment` ima 9 vrednosti (uključujući nedostajuće) — segmenti su standardne oznake (A, B, C, D, E, F, J, M, S).
- `make` ima 96 jedinstvenih vrednosti, `model` još više — ovo su visoko-kardinalne kategorijske kolone i moraćemo posebno da razmislimo kako ih kodirati (one-hot encoding bi napravio previše kolona).
- Nema problema sa velikim/malim slovima ili viškom razmaka — vrednosti izgledaju dosledno zapisane.

## Provera naziva kolona

In [23]:
df.columns.tolist()

['make',
 'model',
 'priceUSD',
 'year',
 'condition',
 'mileage(kilometers)',
 'fuel_type',
 'volume(cm3)',
 'color',
 'transmission',
 'drive_unit',
 'segment']

**Zaključak:** Kolone `mileage(kilometers)` i `volume(cm3)` sadrže zagrade, što nije praktično za rad u Pythonu (pravimo problem sa `df.mileage(kilometers)` sintaksom). Ove nazive treba standardizovati u snake_case bez specijalnih znakova (npr. `mileage_km`, `volume_cm3`) u fazi čišćenja.

## Spisak problema za fazu čišćenja

Na osnovu ove analize, u fazi čišćenja ćemo rešavati sledeće probleme:

1. **Nazivi kolona** — `mileage(kilometers)` i `volume(cm3)` sadrže zagrade i treba ih standardizovati u snake_case (npr. `mileage_km`, `volume_cm3`).
2. **Duplikati** — 87 potpuno identičnih redova treba ukloniti.
3. **Nedostajuće vrednosti** — `volume(cm3)` (47), `drive_unit` (1905) i `segment` (5291) sadrže nedostajuće vrednosti koje treba obraditi (imputacija ili posebna kategorija "unknown").
4. **Nevalidna kilometraža** — 364 reda imaju kilometražu preko 1.000.000 km, što je nerealno; ove redove treba ili ukloniti ili tretirati kao nevalidne.
5. **Ekstremno stare godine** — 124 automobila starija od 1970. godine; treba odlučiti da li ih zadržati.
6. **Ekstremne cene** — 33 automobila sa cenom preko 100.000$; treba proveriti da li su validni ili outlieri.
7. **Ekstremna zapremina motora** — maksimalna vrednost od 20000 cm³ deluje kao outlier za putnički automobil.
8. **Visoko-kardinalne kategorijske kolone** — `make` (96 vrednosti) i `model` (mnogo više) zahtevaju poseban pristup kodiranju, jer standardni one-hot encoding bi napravio previše kolona.

Ovaj spisak nam služi kao osnova za narednu lekciju/korak, u kojoj ćemo napisati skript za čišćenje podataka (`src/data_cleaning.py`).